# ARC-AGI Solver — Kaggle Submission

Generates `submission.json` from a trained checkpoint.

## One-time setup before running this notebook

You need two Kaggle datasets attached to this notebook:

### Dataset 1 — Model checkpoint
1. Download `transformer_call_400_arc_best.pt` from your Google Drive (`arc-agi/checkpoints/`)
2. Kaggle → Datasets → **New Dataset** → upload the `.pt` file → name it `arc-agi-model`

### Dataset 2 — Source code
On your local machine, run this once to create a zip:
```bash
cd ~/Documents/GitHub/arc-agi-solver
zip -r /tmp/arc-agi-code.zip src/ scripts/evaluate.py scripts/submit.py
```
Then: Kaggle → Datasets → **New Dataset** → upload `arc-agi-code.zip` → name it `arc-agi-code`

### Add inputs to this notebook
In the Kaggle notebook editor, click **+ Add data** and attach:
- The ARC Prize competition dataset (search `arc-prize-2024`)
- Your `arc-agi-model` dataset
- Your `arc-agi-code` dataset

Then **Run All** → **Submit**.

---
**Cell order:** 1 (GPU check) → 2 (find paths) → 3 (config) → 4 (prep data) → 5 (run inference) → 6 (verify)

In [ ]:
# ── Cell 1: Check GPU and list available inputs ──────────────────────────────
import torch
from pathlib import Path

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — enable GPU accelerator in notebook settings.')

print('\nAvailable input datasets:')
for d in sorted(Path('/kaggle/input').iterdir()):
    files = sorted(d.rglob('*'))[:6]
    print(f'  {d.name}/')
    for f in files:
        if f.is_file():
            print(f'    {f.relative_to(d)}  ({f.stat().st_size/1e6:.1f} MB)')

In [ ]:
# ── Cell 2: Locate code, checkpoint, and competition data ────────────────────
import sys
import zipfile
from pathlib import Path

WORKING = Path('/kaggle/working')
INPUT   = Path('/kaggle/input')

# ── Unzip code dataset if needed ─────────────────────────────────────────────
code_root = WORKING / 'arc-agi-code'
if not code_root.exists():
    # Find the zip in any input dataset
    zips = list(INPUT.rglob('arc-agi-code.zip'))
    if zips:
        print(f'Extracting code from {zips[0]} ...')
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(WORKING)
        # After extraction the directory may be named arc-agi-code/
        # or sit one level deeper — find it
        candidates = [d for d in WORKING.rglob('src') if d.is_dir()]
        if candidates:
            code_root = candidates[0].parent
        print(f'Code extracted to: {code_root}')
    else:
        # Code is already a directory in an input dataset (uploaded unzipped)
        for d in INPUT.iterdir():
            if (d / 'src').exists():
                code_root = d
                print(f'Code found at: {code_root}')
                break
        else:
            raise RuntimeError(
                'arc-agi-code dataset not found.\n'
                'Add the arc-agi-code dataset to this notebook (see cell 0 instructions).'
            )

if str(code_root) not in sys.path:
    sys.path.insert(0, str(code_root))
print(f'Code root: {code_root}  (src/ exists: {(code_root / "src").exists()})')

# ── Find checkpoint (.pt file) ────────────────────────────────────────────────
pt_files = sorted(INPUT.rglob('*.pt'))
if not pt_files:
    raise RuntimeError(
        'No .pt checkpoint found.\n'
        'Add the arc-agi-model dataset to this notebook (see cell 0 instructions).'
    )
CKPT_PATH = str(pt_files[0])
print(f'Checkpoint: {CKPT_PATH}  ({Path(CKPT_PATH).stat().st_size/1e6:.1f} MB)')
if len(pt_files) > 1:
    print(f'  (multiple checkpoints found — using first; edit CKPT_PATH to change)')
    for p in pt_files:
        print(f'    {p}')

# ── Find competition test challenges JSON ─────────────────────────────────────
# ARC Prize 2024 stores data in arc-prize-2024/ with files like:
#   arc-agi_test_challenges.json
#   arc-agi_evaluation_challenges.json
#   arc-agi_evaluation_solutions.json
test_json = None
for name in ('arc-agi_test_challenges.json', 'arc-agi_evaluation_challenges.json'):
    matches = list(INPUT.rglob(name))
    if matches:
        test_json = matches[0]
        break
if test_json is None:
    # Fall back: any JSON with 'test' in name
    matches = [f for f in INPUT.rglob('*.json')
               if 'test' in f.name.lower() or 'challenge' in f.name.lower()]
    if matches:
        test_json = matches[0]

if test_json is None:
    raise RuntimeError(
        'Competition test data not found.\n'
        'Add the arc-prize-2024 competition dataset to this notebook.'
    )
print(f'Test challenges: {test_json}  ({test_json.stat().st_size/1e3:.0f} KB)')

# Check for solutions (evaluation mode — gives accuracy without submitting)
solutions_json = None
for name in ('arc-agi_evaluation_solutions.json', 'arc-agi_test_solutions.json'):
    matches = list(INPUT.rglob(name))
    if matches:
        solutions_json = matches[0]
        break
print(f'Solutions: {solutions_json}  (accuracy measurement {'available' if solutions_json else 'not available — blind submission'})')

In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────────────────
# These are the only settings you might want to change.

K_CONTEXT = 3    # context pairs fed to the model (must match training)
N_PERMS   = 20   # colour permutations for TTA (more = more accurate, slower)
N_D4      = 8    # D4 orientations: 1=colour-only, 8=all geometric symmetries
SEED1     = 42   # RNG seed for attempt 1
SEED2     = 137  # RNG seed for attempt 2 (independent — maximises coverage)

# Override checkpoint path here if needed (auto-detected above)
# CKPT_PATH = '/kaggle/input/arc-agi-model/transformer_call_400_arc_best.pt'

SUBMISSION_PATH = str(WORKING / 'submission.json')

print(f'Checkpoint:  {CKPT_PATH}')
print(f'Test data:   {test_json}')
print(f'Output:      {SUBMISSION_PATH}')
print(f'TTA:         n_perms={N_PERMS}  n_d4={N_D4}  k_ctx={K_CONTEXT}')

In [ ]:
# ── Cell 4: Extract test tasks to individual files ───────────────────────────
# submit.py reads from a directory of per-task JSON files.
# Kaggle provides one combined JSON — split it here.
# Also merge solutions if available (enables local accuracy measurement).
import json
from pathlib import Path

DATA_DIR = WORKING / 'data' / 'test'
DATA_DIR.mkdir(parents=True, exist_ok=True)

with open(test_json) as f:
    challenges = json.load(f)

solutions = {}
if solutions_json is not None:
    with open(solutions_json) as f:
        solutions = json.load(f)

n_written = 0
for tid, task in challenges.items():
    # Merge solutions into test pairs if available (for accuracy measurement)
    if tid in solutions:
        for i, output in enumerate(solutions[tid]):
            if i < len(task['test']):
                task['test'][i]['output'] = output
    (DATA_DIR / f'{tid}.json').write_text(json.dumps(task))
    n_written += 1

print(f'Wrote {n_written} task files to {DATA_DIR}')
if solutions:
    print(f'Solutions merged: accuracy will be computed ({len(solutions)} tasks have answers)')
else:
    print('No solutions — blind submission (accuracy metric will show 0)')

In [ ]:
# ── Cell 5: Run inference and write submission.json ──────────────────────────
import subprocess
import sys
from pathlib import Path

submit_script = str(code_root / 'scripts' / 'submit.py')

cmd = [
    sys.executable, '-u', submit_script,
    '--checkpoint',  CKPT_PATH,
    '--data-dir',    str(DATA_DIR),
    '--n-perms',     str(N_PERMS),
    '--n-d4',        str(N_D4),
    '--k-context',   str(K_CONTEXT),
    '--seed',        str(SEED1),
    '--seed2',       str(SEED2),
    '--output-file', SUBMISSION_PATH,
    '--verbose',
]

print(f'Running inference on {n_written} tasks...')
print('=' * 60)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('=' * 60)
print(f'Exit code: {proc.returncode}')

In [ ]:
# ── Cell 6: Verify submission ────────────────────────────────────────────────
import json
from pathlib import Path

sub_path = Path(SUBMISSION_PATH)
if not sub_path.exists():
    print('ERROR: submission.json was not created — check Cell 5 output for errors.')
else:
    with open(sub_path) as f:
        sub = json.load(f)

    n_tasks   = len(sub)
    n_attempt1 = sum(1 for v in sub.values() if 'attempt_1' in v)
    n_attempt2 = sum(1 for v in sub.values() if 'attempt_2' in v)
    sz_kb      = sub_path.stat().st_size / 1e3

    print(f'submission.json — {sz_kb:.0f} KB')
    print(f'  Tasks:     {n_tasks}')
    print(f'  attempt_1: {n_attempt1}/{n_tasks}')
    print(f'  attempt_2: {n_attempt2}/{n_tasks}')

    # Show a few sample entries
    print('\nSample entries:')
    for i, (tid, pred) in enumerate(list(sub.items())[:3]):
        a1 = pred.get('attempt_1', [])
        a2 = pred.get('attempt_2', [])
        print(f'  {tid}: attempt_1 shape={len(a1)}×{len(a1[0]) if a1 else 0}  '
              f'attempt_2 shape={len(a2)}×{len(a2[0]) if a2 else 0}')

    if n_tasks == n_attempt1 == n_attempt2:
        print(f'\n✓ Submission looks complete — ready to submit.')
    else:
        print(f'\nWARNING: some tasks are missing predictions.')